## **Feature Analysis**

    Analysis of data/final/train_final.csv 
    (4,184 rows, 19 features + price) before 
    feature selection was applied. Target is log1p(price) 
    throughout, since raw price is heavily right-skewed 
    (median ~$14k, max ~$3.9M).

In [16]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr, f_oneway, chi2_contingency
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("china_used_cars.csv")
y = df["price"]
y_log = np.log1p(y)
X = df.drop(columns=["price"])

numeric_cols = [
    "mileage_km", "engine_cc", "model_year", "seat_count", "motor_power_kw",
    "battery_capacity_kwh", "door_count", "year", "month", "car_age",
    "mileage_per_year", "quarter", "log_mileage",
]
cat_cols = ["fuel_type", "level", "car_body_color", "drive_mode",
            "transmission_Manual", "is_electric"]

## **1. Correlation with target**

In [17]:
print("=== Numeric vs price (log): Pearson & Spearman ===")
for c in numeric_cols:
    pr = pearsonr(df[c], y_log)[0]
    sr = spearmanr(df[c], y_log)[0]
    print(f"{c:22s} pearson={pr:+.3f}  spearman={sr:+.3f}")

print("\n=== Categorical vs price (log): ANOVA F-test ===")
for c in cat_cols:
    groups = [y_log[df[c] == v] for v in df[c].unique()]
    groups = [g for g in groups if len(g) > 1]
    f, p = f_oneway(*groups)
    print(f"{c:22s} F={f:10.2f}  p={p:.4g}")

print("\n=== Categorical vs price-quartile: Chi-square ===")
# chi-square needs two categorical variables, so price is binned into
# quartiles first - this is an approximation, not a substitute for ANOVA
price_bin = pd.qcut(y, q=4, labels=False, duplicates="drop")
for c in cat_cols:
    ct = pd.crosstab(df[c], price_bin)
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"{c:22s} chi2={chi2:10.2f}  p={p:.4g}")

=== Numeric vs price (log): Pearson & Spearman ===
mileage_km             pearson=-0.429  spearman=-0.493
engine_cc              pearson=+0.463  spearman=+0.289
model_year             pearson=+0.409  spearman=+0.494
seat_count             pearson=-0.067  spearman=+0.081
motor_power_kw         pearson=+0.348  spearman=+0.377
battery_capacity_kwh   pearson=+0.236  spearman=+0.305
door_count             pearson=-0.152  spearman=+0.018
year                   pearson=+0.258  spearman=+0.492
month                  pearson=-0.102  spearman=-0.086
car_age                pearson=+0.017  spearman=-0.056
mileage_per_year       pearson=-0.379  spearman=-0.450
quarter                pearson=-0.097  spearman=-0.084
log_mileage            pearson=-0.365  spearman=-0.493

=== Categorical vs price (log): ANOVA F-test ===
fuel_type              F=     88.70  p=4.752e-105
level                  F=    223.94  p=0
car_body_color         F=     82.78  p=5.077e-52
drive_mode             F=    178.85  p=1.038

car_age is essentially uncorrelated with price — irrelevant on its own. door_count's sign flips between Pearson and Spearman — no reliable monotonic relationship

All significant at this sample size — judge by F/Chi² magnitude, not p-value. car_body_color is the weakest of the group.

## **2. Correlation with other features (redundancy)**

In [18]:
corr = X.corr().abs()
print("=== Feature pairs with |corr| > 0.7 ===")
for i, a in enumerate(corr.columns):
    for b in corr.columns[i + 1:]:
        if corr.loc[a, b] > 0.7:
            print(f"{a:22s} <-> {b:22s}  corr={corr.loc[a, b]:.3f}")

=== Feature pairs with |corr| > 0.7 ===
mileage_km             <-> mileage_per_year        corr=0.883
mileage_km             <-> log_mileage             corr=0.727
fuel_type              <-> battery_capacity_kwh    corr=0.773
fuel_type              <-> is_electric             corr=0.879
motor_power_kw         <-> battery_capacity_kwh    corr=0.784
motor_power_kw         <-> is_electric             corr=0.728
battery_capacity_kwh   <-> is_electric             corr=0.841
month                  <-> quarter                 corr=0.970


is_electric is redundant with three other features at once — it was derived from them, so keeping all four adds no new information.

## **3. Feature importance: tree vs linear model**

In [19]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y_log)
rf_importance = pd.Series(rf.feature_importances_, index=X.columns) \
    .sort_values(ascending=False)
print("=== RandomForest importances ===")
print(rf_importance)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge()
ridge.fit(X_scaled, y_log)
ridge_coef = pd.Series(np.abs(ridge.coef_), index=X.columns) \
    .sort_values(ascending=False)
print("\n=== Ridge |standardized coefficients| ===")
print(ridge_coef)

=== RandomForest importances ===
engine_cc               0.233236
level                   0.197003
model_year              0.170701
drive_mode              0.063437
motor_power_kw          0.056366
mileage_km              0.051101
seat_count              0.049496
log_mileage             0.039995
year                    0.034945
transmission_Manual     0.027869
mileage_per_year        0.021097
door_count              0.014646
fuel_type               0.010426
month                   0.009194
battery_capacity_kwh    0.008919
car_body_color          0.004926
quarter                 0.003057
car_age                 0.002919
is_electric             0.000666
dtype: float64

=== Ridge |standardized coefficients| ===
engine_cc               0.493267
motor_power_kw          0.424457
model_year              0.410450
fuel_type               0.294824
transmission_Manual     0.224764
mileage_km              0.191628
seat_count              0.120494
level                   0.113967
car_age           

Where RF and Ridge disagree is the useful signal:

    level — high in RF, low in Ridge → its effect on price is nonlinear (threshold-like), which only a tree can capture.
    fuel_type — low in RF, high in Ridge → its effect is fairly monotonic/linear, so the tree gets little extra benefit from splitting on it.
    car_age — high Ridge coefficient despite ~0 real correlation and ~0 RF importance is a multicollinearity artifact (overlaps with model_year/year) — trust RF, not Ridge, here.

## **4. Performance impact (train with/without each feature)**

In [20]:
def cv_score(X, y_log, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses, r2s = [], []
    for train_idx, val_idx in cv.split(X):
        model = RandomForestRegressor(random_state=42)
        model.fit(X.iloc[train_idx], y_log.iloc[train_idx])
        pred = np.expm1(model.predict(X.iloc[val_idx]))
        y_val = np.expm1(y_log.iloc[val_idx])
        rmses.append(mean_squared_error(y_val, pred) ** 0.5)
        r2s.append(r2_score(y_val, pred))
    return np.mean(rmses), np.mean(r2s)

base_rmse, base_r2 = cv_score(X, y_log)
print(f"ALL 19 features: RMSE=${base_rmse:,.0f} R2={base_r2:.3f}")

candidates = ["is_electric", "car_age", "quarter", "mileage_per_year", "car_body_color"]
for c in candidates:
    rmse, r2 = cv_score(X.drop(columns=[c]), y_log)
    print(f"without {c:20s}: RMSE=${rmse:,.0f} R2={r2:.3f}  (delta R2 = {r2 - base_r2:+.4f})")

rmse, r2 = cv_score(X.drop(columns=candidates), y_log)
print(f"without all 5: RMSE=${rmse:,.0f} R2={r2:.3f}  (delta R2 = {r2 - base_r2:+.4f})")

ALL 19 features: RMSE=$73,495 R2=0.610
without is_electric         : RMSE=$72,847 R2=0.618  (delta R2 = +0.0080)
without car_age             : RMSE=$73,004 R2=0.615  (delta R2 = +0.0053)
without quarter             : RMSE=$73,802 R2=0.607  (delta R2 = -0.0033)
without mileage_per_year    : RMSE=$72,750 R2=0.620  (delta R2 = +0.0095)
without car_body_color      : RMSE=$73,301 R2=0.613  (delta R2 = +0.0032)
without all 5: RMSE=$71,934 R2=0.629  (delta R2 = +0.0189)


    Every one of these features helps to remove individually,
    and removing them together compounds to a bigger gain — confirming 
    they're net noise, not just individually weak signal that combines usefully.

    Conclusion

    is_electric, car_age, quarter, mileage_per_year, car_body_color are irrelevant/redundant 
    and are dropped. level and fuel_type are real signal but nonlinear in opposite directions, 
    which is the main reason RandomForest (R² 0.632) beats Linear/Ridge (R² 0.307) on this dataset.